In [ ]:
import logging
import re
from functools import partial

import numpy as np
import torch
import transformer_lens
from datasets import load_dataset
from tqdm import tqdm
from transformer_lens import HookedTransformer
from transformer_lens.utils import get_act_name

from group_sae.hooks import from_tokens
from group_sae.utils import get_device_for_block, load_cluster_map, load_saes

In [ ]:
def sae_hook(act, hook, sae, cache):
    original_shape = act.shape
    if len(original_shape) == 4:
        x = act.reshape(act.shape[0], act.shape[1], -1).clone()
    else:
        x = act.clone()
    x = x.to(sae.device)

    f = sae.encode(x)
    x_hat = sae.decode(f)

    if torch.is_grad_enabled():
        f.retain_grad()

    residual = x - x_hat
    cache[hook.name] = f

    x_recon = x_hat + residual.detach()

    if len(original_shape) == 4:
        return x_recon.reshape(original_shape)

    return x_recon

### Parameters

In [ ]:
model_name = "pythia-160m"
n_devices = 2
batch_size = 4
layer = None
component = "resid_post"
sae_root_folder = "/home/fbelotti/group-sae/saes/pythia_160m-topk"
K = 3

In [ ]:
if n_devices > 1:
    transformer_lens.utilities.devices.get_device_for_block_index = get_device_for_block

### SAEs loading

In [ ]:
model = HookedTransformer.from_pretrained(model_name, device="cuda", n_devices=n_devices)
device = model.cfg.device
if device is None:
    device = "cuda"
    model.cfg.device = device
if layer is None:
    layers = list(range(model.cfg.n_layers - 1))
else:
    layers = [layer]
modules = [get_act_name(component, layer) for layer in layers]
cluster = K != -1
dictionaries = load_saes(
    sae_root_folder,
    device=device,
    debug=True,
    layer=layer,
    cluster=None if K == -1 else str(K),
    load_from_sae_lens=False,
    dtype="float32",
    model_name=model_name,
)
dictionaries = {
    k: v.to(get_device_for_block(int(re.findall(r"\d+", k)[0]), model.cfg, device=device))
    for k, v in dictionaries.items()
}
if len(dictionaries) == 0:
    raise ValueError("No dictionaries were loaded. Check the path to the dictionaries.")
elif len(dictionaries) != len(modules):
    logging.warning(
        f"Loaded {len(dictionaries)} dictionaries, but expected {len(modules)}. "
        "Some modules may not have been loaded."
    )
    modules = [k for k in modules if k in dictionaries.keys()]
    dictionaries = {k: v for k, v in dictionaries.items() if k in modules}

### Activation caching

In [ ]:
counter_mat = torch.zeros(
    (len(dictionaries), dictionaries[modules[0]].cfg.d_sae),
    dtype=torch.int64,
    device="cuda:0",
)

In [ ]:
# Load dataset
dataset = load_dataset(
    "NeelNanda/pile-small-tokenized-2b", streaming=False, split="train"
).shuffle(seed=42)
dataset = dataset.select(range(len(dataset) // 2, len(dataset)))

In [ ]:
dl = torch.utils.data.DataLoader(
    dataset,
    batch_size=batch_size,
    collate_fn=from_tokens,
    num_workers=8,
    pin_memory=True,
)

In [ ]:
hooks = []
feature_cache = {}
for hook_name in dictionaries.keys():
    hooks.append((hook_name, partial(sae_hook, sae=dictionaries[hook_name], cache=feature_cache)))

In [ ]:
processed_tokens = 0
max_tokens = 1_000_000
for tokens in tqdm(dl, total=max_tokens // (1024 * batch_size)):
    if processed_tokens >= max_tokens:
        break
    with torch.no_grad():
        model.run_with_hooks(tokens["input_ids"].to(device), fwd_hooks=hooks)
    for hook_name, features in feature_cache.items():
        _, features_idxes = torch.topk(features, k=128, dim=-1)
        features_idxes = features_idxes.view(-1)
        layer = int(re.findall(r"\d+", hook_name)[0])
        counter_mat[layer] += torch.bincount(features_idxes, minlength=counter_mat.shape[1]).to(
            "cuda:0"
        )
    feature_cache.clear()
    processed_tokens += tokens["input_ids"].numel()

In [ ]:
cluster_map = load_cluster_map("160m")
cluster_ids = cluster_map[str(K)]
cluster_ids

In [ ]:
# Normnalize the counter matrix by groups counts
plot_mat = torch.zeros_like(counter_mat).float()

for cid in np.unique(cluster_ids):
    mask = np.array(cluster_ids) == cid
    sub_mat = counter_mat[mask].float()
    sub_mat /= sub_mat.sum(dim=0, keepdim=True)

    # Sorting by layer frequency
    weight = (
        torch.arange(sub_mat.shape[0], device=sub_mat.device).float().unsqueeze(1) * sub_mat
    ).sum(0)
    _, idx = torch.sort(weight, descending=False)
    plot_mat[mask] = sub_mat[:, idx]

plot_mat = plot_mat.cpu().numpy()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

fig, ax = plt.subplots(figsize=(7, 2.5), dpi=150)

palette = ["#FFFFFF","#FFC533", "#f48c06", "#DD5703", "#d00000", "#6A040F"]
cmap = LinearSegmentedColormap.from_list("paper", palette)

# Create the heatmap
sns.heatmap(plot_mat, cmap=cmap, annot=False, ax=ax)

# Remove x-ticks (as in your original code) but keep y-ticks if you wish
ax.set_xticks([])
ax.set_xlabel("Sorted features")
ax.set_ylabel("Layers")
ax.set_title(f"Group-SAE G={K} - {model_name.title()}")

# 1. Add a border around the heatmap
for _, spine in ax.spines.items():
    spine.set_visible(True)
    spine.set_linewidth(1)
    spine.set_color("black")

# 2. Add horizontal lines where cluster_ids change
# Find the row indices where the cluster ID changes
cluster_changes = []
for i in range(1, len(cluster_ids)):
    if cluster_ids[i] != cluster_ids[i - 1]:
        cluster_changes.append(i)

# The number of columns in the heatmap is half of plot_mat.shape[1] (since we used ::2)
n_cols = plot_mat.shape[1]

# Draw horizontal lines
for row in cluster_changes:
    ax.hlines(y=row, xmin=0, xmax=n_cols, color="black", linewidth=1)

plt.tight_layout()
plt.show()